# FINN-compatible QONNX Export Script for Road Sign Classifier

This notebook contains the code for exporting a trained quantized model to QONNX format for FINN deployment.

In [1]:
#!/usr/bin/env python3
# Path: models/road_sign_classifier.py
"""
FINN-compatible QONNX export script for minimal 3-block road sign classifier.
This script exports a trained quantized model to QONNX format for FINN deployment.
"""

import os
import torch
import torch.nn as nn
import brevitas.nn as qnn
from brevitas.quant.scaled_int import Int8WeightPerTensorFloat, Int8ActPerTensorFloat
from brevitas.core.zero_point import ZeroZeroPoint
from brevitas.export import export_qonnx
import pickle
import json
import argparse

In [2]:
# Path: models/quantizers.py
# --- FINN-Compatible Quantizers ---
class FINNUint4ActQuant(Int8ActPerTensorFloat):
    """
    FINN-compatible unsigned 4-bit activation quantizer for ReLU.
    CRITICAL: FINN requires UNSIGNED and NON-NARROW activations for ReLU.
    """
    bit_width = 4
    signed = False  # UNSIGNED for FINN ReLU compatibility
    narrow = False  # NON-NARROW for FINN compatibility
    zero_point_impl = ZeroZeroPoint
    
class FINNInt4IdentityQuant(Int8ActPerTensorFloat):
    """
    FINN-compatible signed 4-bit activation quantizer for Identity.
    CRITICAL: FINN requires SIGNED quantization for identity activations.
    """
    bit_width = 4
    signed = True  # SIGNED for FINN Identity compatibility
    narrow = False  # NON-NARROW for FINN compatibility
    zero_point_impl = ZeroZeroPoint
    
class FINNInt4WeightQuant(Int8WeightPerTensorFloat):
    """FINN-compatible 4-bit signed weight quantizer."""
    bit_width = 4
    signed = True  # Weights can be signed
    narrow = False  # NON-NARROW for FINN compatibility

In [3]:
# Path: models/network.py
# --- Minimal Model Definition (FINN-Friendly) ---
class MinimalRoadSignNet(nn.Module):
    """
    Minimal FINN-compatible W4A4 road sign classifier.
    Exactly 3 blocks with 3 layers each.
    """
    
    def __init__(self, num_classes=43):
        super(MinimalRoadSignNet, self).__init__()
        
        # Use FINN-compatible quantizers
        self.weight_quant = FINNInt4WeightQuant
        self.relu_quant = FINNUint4ActQuant  # Unsigned for ReLU
        self.identity_quant = FINNInt4IdentityQuant  # Signed for Identity
        
        # Input quantization - ESSENTIAL for FINN - MUST BE SIGNED
        self.input_quant = qnn.QuantIdentity(
            act_quant=self.identity_quant,  # Use SIGNED identity quantizer
            return_quant_tensor=True
        )
        
        # BLOCK 1: Input -> 16 channels (3 layers)
        self.block1 = nn.Sequential(
            # Layer 1: Conv
            qnn.QuantConv2d(3, 16, kernel_size=3, stride=2, padding=1, 
                            weight_quant=self.weight_quant, bias=False),
            # Layer 2: BatchNorm
            nn.BatchNorm2d(16),
            # Layer 3: ReLU
            qnn.QuantReLU(act_quant=self.relu_quant, return_quant_tensor=True)
        )
        
        # BLOCK 2: 16 -> 32 channels (3 layers)
        self.block2 = nn.Sequential(
            # Layer 1: Conv
            qnn.QuantConv2d(16, 32, kernel_size=3, stride=2, padding=1, 
                           weight_quant=self.weight_quant, bias=False),
            # Layer 2: BatchNorm
            nn.BatchNorm2d(32),
            # Layer 3: ReLU
            qnn.QuantReLU(act_quant=self.relu_quant, return_quant_tensor=True)
        )
        
        # BLOCK 3: 32 -> 64 channels (3 layers)
        self.block3 = nn.Sequential(
            # Layer 1: Conv
            qnn.QuantConv2d(32, 64, kernel_size=3, stride=2, padding=1, 
                           weight_quant=self.weight_quant, bias=False),
            # Layer 2: BatchNorm
            nn.BatchNorm2d(64),
            # Layer 3: ReLU
            qnn.QuantReLU(act_quant=self.relu_quant, return_quant_tensor=True)
        )
        
        # Classifier (fully connected layers)
        self.classifier = nn.Sequential(
            qnn.QuantLinear(64 * 4 * 4, 128, bias=True, weight_quant=self.weight_quant),
            qnn.QuantReLU(act_quant=self.relu_quant, return_quant_tensor=True),
            qnn.QuantLinear(128, num_classes, bias=True, weight_quant=self.weight_quant)
        )
        
        self.quant_mode = 'w4a4'
        self.num_classes = num_classes

    def forward(self, x):
        # Input quantization
        x = self.input_quant(x)
        
        # Handle QuantTensor conversion if needed
        if hasattr(x, 'value'):
            x = x.value
        
        # Process through blocks
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        
        # Flatten and classify
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        
        return x

In [4]:
# Path: models/folding_config.py
def create_folding_config(output_dir):
    """Create a FINN-compatible folding configuration for the minimal model."""
    folding_config = {
        "Defaults": {
            "SIMD": [2, ["StreamingFCLayer_Batch", "MVAU"]],
            "PE": [4, ["StreamingFCLayer_Batch", "MVAU"]],
            "ram_style": ["block", ["StreamingFCLayer_Batch", "MVAU"]],
            "resType": ["lut", ["StreamingFCLayer_Batch", "MVAU"]]
        },
        # Block 1 layers
        "StreamingFCLayer_0": {
            "PE": 4,
            "SIMD": 3,
            "ram_style": "distributed"
        },
        # Block 2 layers
        "StreamingFCLayer_1": {
            "PE": 4,
            "SIMD": 4,
            "ram_style": "block"
        },
        # Block 3 layers
        "StreamingFCLayer_2": {
            "PE": 8,
            "SIMD": 8,
            "ram_style": "block"
        },
        # Classifier layers
        "StreamingFCLayer_3": {
            "PE": 16,
            "SIMD": 16,
            "ram_style": "block"
        },
        # MVAU nodes
        "MVAU_hls_0": {
            "SIMD": 3,
            "PE": 4
        },
        "MVAU_hls_1": {
            "SIMD": 4,
            "PE": 4
        },
        "MVAU_hls_2": {
            "SIMD": 8,
            "PE": 8
        },
        # FIFO depths - keep small to reduce resource usage
        "StreamingFIFO_0": {"depth": 64},
        "StreamingFIFO_1": {"depth": 64},
        "StreamingFIFO_2": {"depth": 128},
        "StreamingFIFO_3": {"depth": 128},
        "StreamingFIFO_4": {"depth": 256},
        "StreamingFIFO_5": {"depth": 256}
    }
    
    # Save folding configuration
    folding_config_path = os.path.join(output_dir, "folding_config_minimal.json")
    with open(folding_config_path, 'w') as f:
        json.dump(folding_config, f, indent=2)
    
    print(f"✅ Folding configuration saved to: {folding_config_path}")
    return folding_config_path

In [5]:
# Path: models/export.py
def export_model(model_path="models/road_sign_minimal_w4a4_finn.pth", 
                output_path="models/road_sign_minimal_finn_w4a4.onnx", 
                dataset_path="dataset/gtsrb_processed.pkl", 
                num_classes=43):
    """Export trained model to QONNX format for FINN."""
    print(f"\n🚀 Exporting minimal 3-block model to QONNX for FINN")
    print("=" * 60)
    
    # Create output directory if needed
    output_dir = os.path.dirname(output_path)
    os.makedirs(output_dir, exist_ok=True)
    
    # Load dataset info to get number of classes if available
    if os.path.exists(dataset_path):
        try:
            with open(dataset_path, 'rb') as f:
                dataset = pickle.load(f)
            num_classes = dataset.get('num_classes', num_classes)
            print(f"📊 Dataset: {num_classes} classes")
        except Exception as e:
            print(f"⚠️  Could not load dataset: {e}")
            print(f"Using provided number of classes: {num_classes}")
    
    # Load trained weights
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    # Create model instance
    model = MinimalRoadSignNet(num_classes=num_classes)
    print(f"✅ Minimal 3-block model created for {num_classes} classes")
    print(f"✅ Using SIGNED quantization for identity activations")
    print(f"✅ Using UNSIGNED quantization for ReLU activations")
    print(f"✅ Each block has exactly 3 layers for maximum FPGA compatibility")
    
    # Load weights
    try:
        # Load checkpoint (handle both direct state_dict and checkpoint format)
        checkpoint = torch.load(model_path, map_location='cpu')
        
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            # Checkpoint format from training
            state_dict = checkpoint['model_state_dict']
            best_val_acc = checkpoint.get('best_val_acc', 'Unknown')
            print(f"📊 Loaded checkpoint with validation accuracy: {best_val_acc}")
        else:
            # Direct state_dict format
            state_dict = checkpoint
            print("📊 Loaded direct state dict")
        
        # Load weights into model
        model.load_state_dict(state_dict)
        model.eval()
        
        print("✅ Model weights loaded successfully")
        
    except Exception as e:
        print(f"❌ Error loading model weights: {e}")
        raise
    
    # Create dummy input (batch_size=1 for FINN)
    dummy_input = torch.randn(1, 3, 32, 32)
    print(f"🔍 Input shape: {dummy_input.shape}")

    # Test forward pass to ensure model works
    print("🧪 Testing forward pass...")
    with torch.no_grad():
        test_output = model(dummy_input)
        print(f"✅ Output shape: {test_output.shape}")
        print(f"✅ Output range: [{test_output.min():.3f}, {test_output.max():.3f}]")
    
    # Export to QONNX (FINN-compatible format)
    print(f"\n🚀 Exporting to QONNX: {output_path}")
    print("This preserves W4A4 quantization information for FINN...")
    
    try:
        export_qonnx(
            model,
            dummy_input,
            export_path=output_path
        )
        
        print("✅ QONNX export successful!")
        
        # Check file size
        file_size = os.path.getsize(output_path) / (1024 * 1024)  # MB
        print(f"📁 File size: {file_size:.2f} MB")
        
    except Exception as e:
        print(f"❌ QONNX export failed: {e}")
        print("\nTrying alternative export methods...")
        
        # Fallback: try with older Brevitas API
        try:
            from brevitas.onnx import export_finn_onnx
            export_finn_onnx(
                model, 
                dummy_input, 
                output_path
            )
            print("✅ Export successful using legacy Brevitas API!")
        except Exception as e2:
            print(f"❌ Legacy export also failed: {e2}")
            raise
    
    # Validate the exported QONNX file
    if os.path.exists(output_path):
        print("\n🔍 Validating QONNX export...")
        
        try:
            import onnx
            
            # Load and check the ONNX model
            onnx_model = onnx.load(output_path)
            onnx.checker.check_model(onnx_model)
            
            print("✅ ONNX model is valid!")
            print(f"📊 IR version: {onnx_model.ir_version}")
            print(f"📊 Opset version: {onnx_model.opset_import[0].version}")
            
            # Count node types
            node_types = {}
            for node in onnx_model.graph.node:
                node_type = node.op_type
                node_types[node_type] = node_types.get(node_type, 0) + 1
            
            print("\n📋 Node types in QONNX model:")
            for op_type, count in sorted(node_types.items()):
                print(f"   {op_type}: {count}")
            
            # Check for quantization nodes
            quant_nodes = ['Quant', 'BinaryQuant', 'Trunc', 'MultiThreshold']
            has_quant = any(op in node_types for op in quant_nodes)
            
            if has_quant:
                print("\n✅ QONNX contains quantization information - FINN ready!")
            else:
                print("\n⚠️  No explicit quantization nodes found")
                
        except ImportError:
            print("⚠️  ONNX package not available for validation")
        except Exception as e:
            print(f"⚠️  Validation error: {e}")
            
    else:
        print("❌ QONNX file was not created successfully")
    
    # Create folding configuration
    folding_config_path = create_folding_config(output_dir)
    
    print("\n🎉 EXPORT COMPLETE!")
    print("=" * 60)
    print(f"✅ QONNX model: {output_path}")
    print(f"✅ Folding config: {folding_config_path}")
    print(f"✅ Architecture: Minimal 3-block CNN (3 layers per block)")
    print(f"✅ Quantization: W4A4 (4-bit weights and activations)")
    print(f"✅ Input shape: (1, 3, 32, 32)")
    print(f"✅ Output classes: {num_classes}")
    
    print("\n📋 NEXT STEPS:")
    print("1. Use the following code in your FINN notebook:")
    print("```python")
    print("from finn.builder.build_dataflow import build_dataflow")
    print("from finn.builder.build_dataflow_config import DataflowBuildConfig, ShellFlowType")
    print("")
    print("# Define the build configuration")
    print("build_cfg = DataflowBuildConfig(")
    print(f"    input_file=\"{os.path.basename(output_path)}\",")
    print("    output_dir=\"finn_output_road_sign\",")
    print("    board=\"Pynq-Z2\",")
    print("    shell_flow_type=ShellFlowType.VIVADO_ZYNQ,")
    print("    synth_clk_period_ns=10.0,")
    print(f"    folding_config_file=\"{os.path.basename(folding_config_path)}\",")
    print("    save_intermediate_models=False  # Set to False to save disk space")
    print(")")
    print("")
    print("# Run the build flow")
    print("build_dataflow(build_cfg)")
    print("```")
    
    return output_path, folding_config_path

## Execute the Export Function

Run the cell below to execute the export function with your desired parameters. You can modify the parameters as needed.

In [6]:
# Path: models/execute.py
# Execute the export function directly
# Modify these parameters as needed for your specific use case

# Create directories if they don't exist
os.makedirs("models", exist_ok=True)
os.makedirs("dataset", exist_ok=True)

# Define parameters
model_path = "rsc_minimal_w4a4_finn.pth"  # Path to your trained model
output_path = "models/road_sign_minimal_finn_w4a4.onnx"  # Where to save the QONNX model
dataset_path = "dataset/gtsrb_processed.pkl"  # Path to your dataset info
num_classes = 43  # Number of classes in your dataset

# Uncomment the line below to execute the export function
output_path, folding_config_path = export_model(model_path, output_path, dataset_path, num_classes)


🚀 Exporting minimal 3-block model to QONNX for FINN
✅ Minimal 3-block model created for 43 classes
✅ Using SIGNED quantization for identity activations
✅ Using UNSIGNED quantization for ReLU activations
✅ Each block has exactly 3 layers for maximum FPGA compatibility
📊 Loaded checkpoint with validation accuracy: 97.93420045906656
✅ Model weights loaded successfully
🔍 Input shape: torch.Size([1, 3, 32, 32])
🧪 Testing forward pass...
✅ Output shape: torch.Size([1, 43])
✅ Output range: [-22.136, 3.808]

🚀 Exporting to QONNX: models/road_sign_minimal_finn_w4a4.onnx
This preserves W4A4 quantization information for FINN...
✅ QONNX export successful!
📁 File size: 0.63 MB

🔍 Validating QONNX export...
✅ ONNX model is valid!
📊 IR version: 7
📊 Opset version: 14

📋 Node types in QONNX model:
   BatchNormalization: 3
   Conv: 3
   Flatten: 1
   Gemm: 2
   Quant: 10
   Relu: 4

✅ QONNX contains quantization information - FINN ready!
✅ Folding configuration saved to: models/folding_config_minimal.j

/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_658/2246203979.py:80: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


## FINN Integration Example

After exporting your model, you can use the following code in a FINN notebook to build the dataflow accelerator.

In [ ]:
# Path: finn/build_example.py
# Example code for FINN integration
# This is for reference only and should be executed in a FINN environment

"""
from finn.builder.build_dataflow import build_dataflow
from finn.builder.build_dataflow_config import DataflowBuildConfig, ShellFlowType

# Define the build configuration
build_cfg = DataflowBuildConfig(
    input_file="road_sign_minimal_finn_w4a4.onnx",
    output_dir="finn_output_road_sign",
    board="Pynq-Z2",
    shell_flow_type=ShellFlowType.VIVADO_ZYNQ,
    synth_clk_period_ns=10.0,
    folding_config_file="folding_config_minimal.json",
    save_intermediate_models=False  # Set to False to save disk space
)

# Run the build flow
build_dataflow(build_cfg)
"""